In [ ]:
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")
print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])

Connected to PostgreSQL: localhost storage_analytics


In [12]:
# Dashboard mart loads
device_overview = pd.read_sql("SELECT * FROM mart_tableau_device_overview", engine)
anomaly_timeline = pd.read_sql("SELECT * FROM mart_tableau_anomaly_timeline", engine)
root_cause_summary = pd.read_sql("SELECT * FROM mart_tableau_root_cause_summary", engine)
grafana_health = pd.read_sql("SELECT * FROM v_grafana_device_health", engine)

datasets = {
    "device_overview": device_overview,
    "anomaly_timeline": anomaly_timeline,
    "root_cause_summary": root_cause_summary,
    "grafana_health": grafana_health,
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows, {df.shape[1]} columns")

device_overview: 5 rows, 13 columns
anomaly_timeline: 2,890 rows, 13 columns
root_cause_summary: 39 rows, 7 columns
grafana_health: 10,080 rows, 12 columns


In [13]:
# Schema and null profile
for name, df in datasets.items():
    print(f"\n{name} columns:")
    print(sorted(df.columns.tolist()))

    null_pct = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
    print("Top null percentages (%):")
    print(null_pct.head(10))


device_overview columns:
['anomaly_count', 'avg_latency_ms', 'avg_queue_depth', 'avg_throughput_mb_s', 'avg_total_iops', 'avg_util_pct', 'critical_anomaly_count', 'device', 'dominant_workload_pattern', 'high_anomaly_count', 'p95_latency_ms', 'p99_latency_ms', 'sample_count']
Top null percentages (%):
device                       0.0
sample_count                 0.0
avg_total_iops               0.0
avg_throughput_mb_s          0.0
avg_latency_ms               0.0
p95_latency_ms               0.0
p99_latency_ms               0.0
avg_util_pct                 0.0
avg_queue_depth              0.0
dominant_workload_pattern    0.0
dtype: float64

anomaly_timeline columns:
['anomaly_score', 'aqu_sz', 'avg_latency_ms', 'detector_type', 'device', 'metric_name', 'root_cause_hint', 'saturation_score', 'severity', 'timestamp', 'total_iops', 'util_pct', 'workload_pattern']
Top null percentages (%):
device              0.0
timestamp           0.0
metric_name         0.0
detector_type       0.0
sever

In [14]:
# Device coverage comparison across datasets
device_sets = {}
for name, df in datasets.items():
    if "device" in df.columns:
        device_sets[name] = set(df["device"].dropna().astype(str).unique())
        print(f"{name}: {len(device_sets[name])} unique devices")
    else:
        device_sets[name] = set()
        print(f"{name}: no device column")

all_devices = sorted(set().union(*device_sets.values()))
print(f"\nAll unique devices across marts: {len(all_devices)}")
print("\n".join(all_devices))

ref_name = "device_overview"
if device_sets.get(ref_name):
    print(f"\nMissing vs {ref_name}:")
    ref = device_sets[ref_name]
    for name, s in device_sets.items():
        if name == ref_name:
            continue
        missing = sorted(ref - s)
        print(f"{name}: {len(missing)} missing")
        if missing:
            print("  sample:", ", ".join(missing[:20]))

device_overview: 5 unique devices
anomaly_timeline: 5 unique devices
root_cause_summary: no device column
grafana_health: 5 unique devices

All unique devices across marts: 5
dm-0
nvme0n1
nvme1n1
sda
sdb

Missing vs device_overview:
anomaly_timeline: 0 missing
root_cause_summary: 5 missing
  sample: dm-0, nvme0n1, nvme1n1, sda, sdb
grafana_health: 0 missing


In [15]:
# Anomaly and root cause quality checks
if "severity" in anomaly_timeline.columns:
    print("Anomaly severity distribution:")
    print(anomaly_timeline["severity"].value_counts(dropna=False))

if "root_cause_hint" in anomaly_timeline.columns:
    print("\nTop root_cause_hint values:")
    print(anomaly_timeline["root_cause_hint"].fillna("<null>").value_counts().head(15))

if "anomaly_count" in root_cause_summary.columns:
    cols = [c for c in ["device", "root_cause_hint", "anomaly_count"] if c in root_cause_summary.columns]
    print("\nTop root cause rows by anomaly_count:")
    print(root_cause_summary.sort_values("anomaly_count", ascending=False)[cols].head(20))

Anomaly severity distribution:
severity
low         2542
high         179
critical     169
Name: count, dtype: int64

Top root_cause_hint values:
root_cause_hint
saturation_score deviated from device baseline                702
iowait_pressure deviated from device baseline                 530
avg_latency_ms deviated from device baseline                  449
queue_efficiency deviated from device baseline                434
total_iops deviated from device baseline                      277
merge_efficiency deviated from device baseline                267
Device saturation with queue buildup                           99
total_throughput_mb_s deviated from device baseline            75
CPU I/O wait pressure indicates backend storage bottleneck     44
Small random I/O pressure reducing merge effectiveness         13
Name: count, dtype: int64

Top root cause rows by anomaly_count:
                                      root_cause_hint  anomaly_count
0        avg_latency_ms deviated from device

In [16]:
# Health/risk screen for dashboard sanity
candidate_cols = ["device", "avg_latency_ms", "util_pct", "aqu_sz", "saturation_score", "anomaly_flag"]
available_cols = [c for c in candidate_cols if c in grafana_health.columns]

health_screen = grafana_health[available_cols].copy()
for c in ["avg_latency_ms", "util_pct", "aqu_sz", "saturation_score"]:
    if c in health_screen.columns:
        health_screen[c] = pd.to_numeric(health_screen[c], errors="coerce")

sort_col = "saturation_score" if "saturation_score" in health_screen.columns else None
if sort_col:
    print("Top 25 rows by saturation_score:")
    print(health_screen.sort_values(sort_col, ascending=False).head(25))

if "anomaly_flag" in health_screen.columns:
    print("\nAnomaly flag distribution:")
    print(health_screen["anomaly_flag"].value_counts(dropna=False))

Top 25 rows by saturation_score:
     device  avg_latency_ms    util_pct     aqu_sz  saturation_score  \
9447    sdb     6310.284724   94.826472  92.694137       8789.858007   
9442    sdb      118.006144   91.608985  48.831489       4473.403139   
9448    sdb      393.387642   82.175214  40.754786       3349.033248   
9449    sdb      554.010629   89.388497  22.584082       2018.757168   
8489    sdb       17.912895  100.000000  19.138384       1913.838353   
8802    sdb       46.793255  100.000000  19.062832       1906.283227   
9069    sdb      136.634718  100.000000  18.980281       1898.028136   
1940   dm-0       10.861321  100.000000  17.044566       1704.456553   
9118    sdb      123.280901  100.000000  15.233759       1523.375884   
9441    sdb      174.308892   90.753771  15.528148       1409.237955   
9114    sdb       21.354699  100.000000  13.817800       1381.779993   
9096    sdb       13.007186  100.000000  13.649071       1364.907126   
8835    sdb       45.381055  10